# 02 - Prepare the synthetic source table

Public research notebook for the STARS workflow. Dataset files and generated artifacts are intentionally excluded from Git. See `../data/README.md` before running.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path("..").resolve()
DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "interim").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_ROOT / "src"))


In [ ]:
import pandas as pd
from pathlib import Path

# 1) Read each CSV and re-format the columns ---------------------------------
root = DATA_DIR / "interim" / "ready-raw"
dfs = []

for csv_path in root.glob("*.csv"):        # picks up merged_67.csv, merged_68.csv
    print("working on", csv_path)
    df = pd.read_csv(csv_path)

    tidy = pd.DataFrame({
        "building_id": csv_path.stem,      # → merged_67 or merged_68
        "timestamp":   df["timestamp"],
        "electricity_kWh":   df["out.electricity.total.energy_consumption"],
        "airTemperature":    df["Dry Bulb Temperature [°C]"],
        "relativeHumidity":  df["Relative Humidity [%]"],
        "windSpeed":         df["Wind Speed [m/s]"],
    })

    # 2) Parse timestamp and add calendar features ----------------------------
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"])
    tidy["month_of_year"] = tidy["timestamp"].dt.month
    tidy["day_of_month"] = tidy["timestamp"].dt.day
    tidy["day_of_week"]  = tidy["timestamp"].dt.dayofweek   # 0 = Monday
    tidy["hour_of_day"]  = tidy["timestamp"].dt.hour

    dfs.append(tidy)

# 3) Concatenate and save -----------------------------------------------------
combined = pd.concat(dfs, ignore_index=True)
combined.to_csv(str(DATA_DIR / "processed" / "combined_ready_raw.csv"), index=False)
print("✓ combined_ready_raw.csv written with", len(combined), "rows")

In [ ]:
combined.head()